In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from collections import deque
import os
import shutil 
import math
import datetime as dt
import random

In [ ]:
from sklearn.model_selection import train_test_split
from keras.layers import *
from keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

In [ ]:
import cv2
import matplotlib.pyplot as plt

def play_video(video_path, n_frames=16):  #name to be cchanged to preview
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError("Couldn't open video")

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.ravel()

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // n_frames, 1)

    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(frame)
        axes[i].axis("off")

    cap.release()
    plt.tight_layout()

In [ ]:
#  classes directories
violence_dir ="../data/violence_fight_detection_dataset/RLVS/train/Fight/"
non_violence_dir ="../data/violence_fight_detection_dataset/RLVS/train/NonFight/"

# retrive list of all the vid files present in the class directory
non_violence_files_names_list = os.listdir(non_violence_dir)
violence_files_names_list = os.listdir(violence_dir)

# randomly select a vid file in from the classes directory
random_nonviolence_vid = random.choice(non_violence_files_names_list)
random_violence_vid = random.choice(violence_files_names_list)

In [ ]:
play_video(f"{non_violence_dir}/{random_nonviolence_vid}")

In [ ]:
play_video(f"{violence_dir}/{random_violence_vid}")

## Extracting Frames

In [ ]:
IMAGE_HEIGHT , IMAGE_WIDTH = 64 , 64

# no of frames of a video that will be fed to the modes as one sequence
SEQUENCE_LENGTH = 16

DATASET_DIR = "../data/violence_fight_detection_dataset/RLVS/train/"

CLASSES_LIST = ['NonFight','Fight']

In [ ]:
import imageio

def frames_extraction(video_path):
    frames_list=[]
    try:
        reader = imageio.get_reader(video_path,"ffmpeg")
        num_frames = reader.count_frames()
        skip_frames_window = max(int(num_frames/SEQUENCE_LENGTH),1)

        for frame_counter in range(0,num_frames,skip_frames_window):
            ''' resizes the frame in the means of heiight & width and also sequence length'''
            frame = reader.get_data(frame_counter)
            frame = cv2.resize(frame,(IMAGE_HEIGHT,IMAGE_WIDTH))
            frame = frame / 255.0
            frames_list.append(frame)
            if len(frames_list) == SEQUENCE_LENGTH:
                break
        reader.close()
    except Exception as e:
        print(f"Error processing {video_path}:{e}")
    return frames_list

In [ ]:
def create_dataset():
    '''provides features , labels , video file paths for for each file  if the sequence lenght is valid'''
    features=[]
    labels=[]
    video_files_paths=[]

    for class_index , class_name in enumerate(CLASSES_LIST):
        print(f'Extracting Data of Class: {class_name}')

        files_list = os.listdir(os.path.join(DATASET_DIR,class_name))

        for file_name in files_list:

            video_file_path = os.path.join(DATASET_DIR,class_name,file_name)

            frames = frames_extraction(video_file_path)

            if len(frames) == SEQUENCE_LENGTH:

                # appending data to their respective lists
                features.append(frames)
                labels.append(class_index)
                video_files_paths.append(video_file_path)

    features = np.asarray(features)
    labels = np.array(labels)
    return features , labels , video_files_paths


In [ ]:
# creating dataset:

features , labels , video_files_paths = create_dataset()

In [ ]:
# saving the extracted data
np.save("features.npy",features)
np.save("labels.npy",labels)
np.save("video_files_paths.npy",video_files_paths)

In [ ]:
# Convert labels into one hot encoded vectors
one_hot_encoded_labels = to_categorical(labels)

In [ ]:
features_train , features_test , labels_train , labels_test = train_test_split(features , one_hot_encoded_labels , test_size = 0.1 , shuffle = True , random_state = 42)

In [ ]:
print(features_train.shape , labels_train.shape)
print(features_test.shape , labels_test.shape)

## Importing and Fine-Tuning MobileNet

In [ ]:
from keras.applications.mobilenet_v2 import MobileNetV2
mobilenet = MobileNetV2(include_top=False , weights='imagenet')

# fine-tuning

mobilenet.trainable=True
for layer in mobilenet.layers[:-40]:
    layer.trainable=False



In [ ]:
mobilenet.summary()

## Building Model

In [ ]:
def create_model():
    model.add(Input(shape=(SEQUENCE_LENGTH , IMAGE_HEIGHT , IMAGE_WIDTH , 3)))

    # passing the mobilenet in the timedistributed layer to handle the sequence
    model.add(TimeDistributed(mobilenet))

    model.add(Dropout(0.25))

    model.add(TimeDistributed(Flatten()))

    lstm_fw = LSTM(units=32)
    lstm_bw = LSTM(units=32 , go_backwards=True)
    
    model.add(Bidirectional(lstm_fw,backward_layer=lstm_bw))

    model.add(Dropout(0.25))
    
    model.add(Dense(256,activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(128,activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(64,activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(32,activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(len(CLASSES_LIST),activation='softmax'))
    
    model.summary()

    return model


In [ ]:
# Constructing the model

MoBiLSTM_model = create_model()

plot_model(MoBiLSTM_model,to_file = 'MoBiLSTM_model_structure_plot.png' , show_shapes=True , show_layer_names=True)

In [ ]:
# specifying callbacks and fittings

early_stopping_callback = EarlyStopping(monitor='val_accuracy', patience = 10 , restore_best_weights=True)

# reducing overfitting by decreasing learning on reduceonplateau callback
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor = 0.6 ,
    patience = 5 , 
    min_lr = 0.00005,
    verbose =1
)

# compiling model
MoBiLSTM_model.compile(loss = 'categorical_crossentropy', optimizer='sgd',metrics=['accuracy'])

# fitting the model
MobBiLSTM_model_history = MoBiLSTM_model.fit(
    x=features_train,
    y=labels_train,
    epochs=20,
    batch_size=8,
    shuffle=True,
    validation_split=0.2,
    callbacks=[early_stopping_callback, reduce_lr]
)